# Store Sales Forecasting

This notebook is my first full time-series forecasting project.

The goal is simple: predict sales for each product family in each Favorita store for the next 16 days. I started with an easy baseline, tested CatBoost, added time-based features, and checked every change with RMSLE (lower is better).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 1. Load and understand the data

Before modeling, I check the files, column types, missing values, duplicates, date range, and unusual values. I keep `sales = 0` because zero sales is a real business result, not a missing value.


In [ ]:
from pathlib import Path

required_files = {
    'train.csv', 'test.csv', 'stores.csv',
    'holidays_events.csv', 'oil.csv', 'sample_submission.csv'
}

data_dirs = [
    path for path in Path('/kaggle/input').iterdir()
    if path.is_dir()
    and required_files.issubset({file.name for file in path.iterdir()})
]

assert len(data_dirs) == 1, f'Expected one competition folder, found {data_dirs}'
data_dir = data_dirs[0]
data_dir


In [ ]:
sorted(file.name for file in data_dir.iterdir())


In [ ]:
train = pd.read_csv(data_dir / "train.csv", parse_dates=["date"])

train.head()

In [ ]:
print("Shape:", train.shape)
train.info()

In [ ]:
summary = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "unique": train.nunique()
})

display(summary)

print("Duplicate rows:", train.duplicated().sum())
print("Duplicate date-store-family:",
      train.duplicated(["date", "store_nbr", "family"]).sum())
print("Date range:", train["date"].min(), "to", train["date"].max())

In [ ]:
display(
    train[["sales", "onpromotion"]]
    .describe(percentiles=[0.5, 0.9, 0.99])
)

print("Negative sales:", (train["sales"] < 0).sum())
print("Negative promotion:", (train["onpromotion"] < 0).sum())
print("Zero sales:", (train["sales"] == 0).sum())
print("Zero sales %:", (train["sales"] == 0).mean() * 100)

In [ ]:
all_dates = pd.date_range(train["date"].min(), train["date"].max())
missing_dates = all_dates.difference(train["date"].unique())

missing_dates

### Data check result

The training data has **3,000,888 rows** from January 2013 to August 2017. I found no missing values and no duplicate store-family-date rows. About **31.3%** of sales values are zero, but they are real observations, so I keep them. The missing calendar dates are Christmas days, not random broken rows.


## 2. Quick EDA

I focus on questions that can change the model:

- Does sales change over time?
- Which weekdays are strongest?
- Does promotion matter?
- Which product families and store types sell the most?


In [ ]:
import matplotlib.pyplot as plt

daily_sales = (
    train.groupby("date", as_index=False)["sales"]
    .sum()
)

plt.figure(figsize=(14, 5))
plt.plot(daily_sales["date"], daily_sales["sales"])
plt.title("Total Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
daily_sales.nlargest(10, "sales")

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

daily_sales["day_name"] = daily_sales["date"].dt.day_name()

weekday_sales = (
    daily_sales.groupby("day_name")["sales"]
    .agg(["mean", "median"])
    .reindex(day_order)
)

display(weekday_sales)

weekday_sales.plot(
    kind="bar",
    figsize=(10, 5),
    title="Average and Median Sales by Day of Week"
)
plt.xlabel("Day of Week")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.show()

In [ ]:
promo_summary = (
    train.assign(has_promotion=train["onpromotion"] > 0)
    .groupby("has_promotion")
    .agg(
        rows=("sales", "size"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        zero_sales_pct=("sales", lambda x: x.eq(0).mean() * 100)
    )
)

promo_summary

In [ ]:
family_sales = (
    train.groupby("family")["sales"]
    .sum()
    .sort_values(ascending=False)
)

family_share = family_sales / family_sales.sum() * 100

display(
    pd.DataFrame({
        "total_sales": family_sales,
        "share_pct": family_share
    }).head(10)
)

family_sales.head(10).sort_values().plot(
    kind="barh",
    figsize=(10, 6),
    title="Top 10 Product Families by Total Sales"
)
plt.xlabel("Total Sales")
plt.show()
print("Top 3 share:", family_share.head(3).sum(), "%")

In [ ]:
stores = pd.read_csv(data_dir / "stores.csv")

display(stores.head())
print("Shape:", stores.shape)
stores.info()

In [ ]:
store_sales = (
    train.groupby("store_nbr", as_index=False)["sales"]
    .sum()
    .merge(stores, on="store_nbr", validate="one_to_one")
)

type_summary = (
    store_sales.groupby("type")
    .agg(
        store_count=("store_nbr", "count"),
        mean_sales_per_store=("sales", "mean"),
        median_sales_per_store=("sales", "median")
    )
    .sort_values("mean_sales_per_store", ascending=False)
)

type_summary

### What I saw

- Saturday and Sunday have the highest sales.
- Rows with promotion have much higher sales.
- Grocery I, Beverages, and Produce are the largest product families.
- Store type A has the highest average sales per store.

These patterns are useful signals, but they do not prove cause and effect.


## 3. Time-based validation

Random split would let future data leak into training, so I use the last 16 days as validation. This matches the 16-day Kaggle test horizon.


In [ ]:
test = pd.read_csv(
    data_dir / "test.csv",
    parse_dates=["date"]
)

print("Test shape:", test.shape)
print("Test date range:", test["date"].min(), "to", test["date"].max())
print("Forecast days:", test["date"].nunique())
display(test.head())

In [ ]:
validation_start = train["date"].max() - pd.Timedelta(days=15)

train_part = train[train["date"] < validation_start].copy()
valid = train[train["date"] >= validation_start].copy()

print("Train:", train_part["date"].min(), "to", train_part["date"].max())
print("Validation:", valid["date"].min(), "to", valid["date"].max())
print("Validation days:", valid["date"].nunique())
print("Validation rows:", len(valid))

### Split result

Training ends on **30 July 2017**. Validation runs from **31 July to 15 August 2017**, giving me 16 future days and 28,512 rows to test on. This setup is much closer to the real Kaggle task than a random split.


## 4. Baseline

The baseline predicts the average sales from the same weekday during the latest 28 days. A model is only useful if it can beat this simple rule.


In [ ]:
from sklearn.metrics import mean_squared_log_error
import numpy as np

baseline_window = train_part[
    train_part["date"] > train_part["date"].max() - pd.Timedelta(days=28)
].copy()

baseline_window["day_of_week"] = baseline_window["date"].dt.dayofweek

baseline_table = (
    baseline_window
    .groupby(["store_nbr", "family", "day_of_week"])["sales"]
    .mean()
    .rename("prediction")
    .reset_index()
)

valid_eval = valid.copy()
valid_eval["day_of_week"] = valid_eval["date"].dt.dayofweek

valid_eval = valid_eval.merge(
    baseline_table,
    on=["store_nbr", "family", "day_of_week"],
    how="left",
    validate="many_to_one"
)

baseline_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_eval["sales"],
        valid_eval["prediction"]
    )
)

print("Missing predictions:", valid_eval["prediction"].isna().sum())
print("Baseline RMSLE:", baseline_rmsle)

### Baseline result

The baseline gets **0.53113 RMSLE**. This is my reference score. A more complex model needs to beat it, otherwise the extra work is not helping.


## 5. CatBoost and a hybrid model

CatBoost can work with categorical columns such as store and product family. I train on `log1p(sales)` because RMSLE cares about relative error and sales is highly skewed.


In [ ]:
model_train = train_part.copy()
model_valid = valid.copy()

for df in [model_train, model_valid]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

features = [
    "store_nbr",
    "family",
    "onpromotion",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "is_weekend"
]

display(model_train[features].head())

In [ ]:
from catboost import CatBoostRegressor

# Use the latest two years to keep this learning notebook practical.
cutoff = model_train["date"].max() - pd.Timedelta(days=730)
fit_data = model_train[model_train["date"] >= cutoff].copy()

cat_features = ["store_nbr", "family", "day_of_week"]

model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

model.fit(
    fit_data[features],
    np.log1p(fit_data["sales"]),
    cat_features=cat_features,
    eval_set=(
        model_valid[features],
        np.log1p(model_valid["sales"])
    ),
    early_stopping_rounds=50
)

valid_prediction = np.expm1(
    model.predict(model_valid[features])
).clip(min=0)

model_rmsle = np.sqrt(
    mean_squared_log_error(
        model_valid["sales"],
        valid_prediction
    )
)

print("Baseline RMSLE:", baseline_rmsle)
print("CatBoost RMSLE:", model_rmsle)

In [ ]:
check = model_valid[
    ["id", "sales", "onpromotion"]
].copy()

check["catboost_prediction"] = valid_prediction

check = check.merge(
    valid_eval[["id", "prediction"]].rename(
        columns={"prediction": "baseline_prediction"}
    ),
    on="id",
    validate="one_to_one"
)

print("Missing:", check.isna().sum().sum())

for has_promotion, group in check.groupby(
    check["onpromotion"] > 0
):
    baseline_score = np.sqrt(
        mean_squared_log_error(
            group["sales"],
            group["baseline_prediction"]
        )
    )

    catboost_score = np.sqrt(
        mean_squared_log_error(
            group["sales"],
            group["catboost_prediction"]
        )
    )

    print(
        "Has promotion:", has_promotion,
        "| Baseline:", baseline_score,
        "| CatBoost:", catboost_score
    )

In [ ]:
check["hybrid_prediction"] = np.where(
    check["onpromotion"] > 0,
    check["catboost_prediction"],
    check["baseline_prediction"]
)

hybrid_rmsle = np.sqrt(
    mean_squared_log_error(
        check["sales"],
        check["hybrid_prediction"]
    )
)

print("Baseline RMSLE:", baseline_rmsle)
print("CatBoost RMSLE:", model_rmsle)
print("Hybrid RMSLE:", hybrid_rmsle)

In [ ]:
check = check.merge(
    model_valid[["id", "date"]],
    on="id",
    validate="one_to_one"
)

split_date = sorted(check["date"].unique())[8]

check["period"] = np.where(
    check["date"] < split_date,
    "First 8 days",
    "Last 8 days"
)

for period, group in check.groupby("period"):
    print("\n", period)

    for name in [
        "baseline_prediction",
        "catboost_prediction",
        "hybrid_prediction"
    ]:
        score = np.sqrt(
            mean_squared_log_error(
                group["sales"],
                group[name]
            )
        )
        print(name, ":", score)

### First result

The plain CatBoost model helped promotional rows but was weak on non-promotional rows. I combined the two methods:

- no promotion: weekday baseline
- promotion: CatBoost

This hybrid reached **0.46769 validation RMSLE** and **0.50406 Kaggle public RMSLE**.


## 6. Extra experiments

Next I tested one change at a time. `sales_lag_28` means sales from the same store and product family exactly 28 calendar days earlier. It is safe for a 16-day forecast because every source value is already known.


In [ ]:
lag_28_source = train_part[
    ["date", "store_nbr", "family", "sales"]
].copy()

lag_28_source["date"] += pd.Timedelta(days=28)

lag_28_source = lag_28_source.rename(
    columns={"sales": "sales_lag_28"}
)

fit_lagged = fit_data.merge(
    lag_28_source,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

valid_lagged = model_valid.merge(
    lag_28_source,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

# GREEN tests
assert "sales_lag_28" in fit_lagged.columns
assert len(fit_lagged) == len(fit_data)
assert len(valid_lagged) == len(model_valid)
assert valid_lagged["sales_lag_28"].notna().all()
assert (
    valid_lagged["date"] - pd.Timedelta(days=28)
    <= train_part["date"].max()
).all()

print("Train missing lag:", fit_lagged["sales_lag_28"].isna().sum())
print("Validation missing lag:", valid_lagged["sales_lag_28"].isna().sum())

display(
    valid_lagged[
        ["date", "store_nbr", "family", "sales", "sales_lag_28"]
    ].head()
)

In [ ]:
features_with_lag = features + ["sales_lag_28"]

lag_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

lag_model.fit(
    fit_lagged[features_with_lag],
    np.log1p(fit_lagged["sales"]),
    cat_features=cat_features,
    eval_set=(
        valid_lagged[features_with_lag],
        np.log1p(valid_lagged["sales"])
    ),
    early_stopping_rounds=50
)

lag_prediction = np.expm1(
    lag_model.predict(valid_lagged[features_with_lag])
).clip(min=0)

lag_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_lagged["sales"],
        lag_prediction
    )
)

print("Old CatBoost:", model_rmsle)
print("CatBoost + lag_28:", lag_rmsle)
print("Old Hybrid:", hybrid_rmsle)

### Lag 28 result

RMSLE drops from **0.55713 to 0.42516**. This is a large improvement, so I keep `sales_lag_28`. It tells the model what happened recently for the exact same store and product family.


### Store information

I add city, state, store type, and cluster, then check whether the validation score improves.


In [ ]:
store_features = ["city", "state", "type", "cluster"]

assert stores["store_nbr"].is_unique

fit_store = fit_lagged.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

valid_store = valid_lagged.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

assert len(fit_store) == len(fit_lagged)
assert len(valid_store) == len(valid_lagged)
assert fit_store[store_features].notna().all().all()
assert valid_store[store_features].notna().all().all()

features_with_store = features_with_lag + store_features
cat_features_with_store = cat_features + store_features

print("Train shape:", fit_store.shape)
print("Validation shape:", valid_store.shape)
display(
    valid_store[
        ["store_nbr", "city", "state", "type", "cluster"]
    ].drop_duplicates().head()
)

In [ ]:
store_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

store_model.fit(
    fit_store[features_with_store],
    np.log1p(fit_store["sales"]),
    cat_features=cat_features_with_store,
    eval_set=(
        valid_store[features_with_store],
        np.log1p(valid_store["sales"])
    ),
    early_stopping_rounds=50
)

store_prediction = np.expm1(
    store_model.predict(valid_store[features_with_store])
).clip(min=0)

store_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_store["sales"],
        store_prediction
    )
)

print("CatBoost + lag_28:", lag_rmsle)
print("CatBoost + lag_28 + stores:", store_rmsle)

### Store feature result

RMSLE improves slightly from **0.42516 to 0.42419**. The gain is small, so these columns add little information beyond the existing features on this validation window. I keep them for the next holiday test.


### Holiday information

National holidays apply everywhere, regional holidays match store state, and local holidays match store city. Transferred holidays are removed from their original date.


In [ ]:
holidays = pd.read_csv(
    data_dir / "holidays_events.csv",
    parse_dates=["date"]
)

print("Shape:", holidays.shape)
print("\nMissing values:")
display(holidays.isna().sum())

print("\nType:")
display(holidays["type"].value_counts())

print("\nLocale:")
display(holidays["locale"].value_counts())

print("\nTransferred:")
display(holidays["transferred"].value_counts())

display(holidays.head(10))

In [ ]:
holidays["date"] = pd.to_datetime(holidays["date"])

active = holidays[~holidays["transferred"]]

holiday_rows = active[
    active["type"].isin(["Holiday", "Additional", "Transfer", "Bridge"])
]
event_rows = active[active["type"] == "Event"]
workday_rows = active[active["type"] == "Work Day"]


def holiday_flag(df, calendar):
    national = df["date"].isin(
        calendar.loc[calendar["locale"] == "National", "date"]
    )

    regional = pd.MultiIndex.from_arrays(
        [df["date"], df["state"]]
    ).isin(
        pd.MultiIndex.from_frame(
            calendar.loc[
                calendar["locale"] == "Regional",
                ["date", "locale_name"]
            ]
        )
    )

    local = pd.MultiIndex.from_arrays(
        [df["date"], df["city"]]
    ).isin(
        pd.MultiIndex.from_frame(
            calendar.loc[
                calendar["locale"] == "Local",
                ["date", "locale_name"]
            ]
        )
    )

    return (national | regional | local).astype("int8")


fit_holiday = fit_store.copy()
valid_holiday = valid_store.copy()

for df in [fit_holiday, valid_holiday]:
    df["is_holiday"] = holiday_flag(df, holiday_rows)
    df["is_event"] = holiday_flag(df, event_rows)
    df["is_work_day"] = holiday_flag(df, workday_rows)

holiday_features = ["is_holiday", "is_event", "is_work_day"]
features_with_holiday = features_with_store + holiday_features

assert len(fit_holiday) == len(fit_store)
assert len(valid_holiday) == len(valid_store)
assert fit_holiday[holiday_features].isin([0, 1]).all().all()
assert valid_holiday[holiday_features].isin([0, 1]).all().all()

print("Train flags:")
display(fit_holiday[holiday_features].sum())

print("Validation flags:")
display(valid_holiday[holiday_features].sum())

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error

holiday_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

holiday_model.fit(
    fit_holiday[features_with_holiday],
    np.log1p(fit_holiday["sales"]),
    cat_features=cat_features_with_store,
    eval_set=(
        valid_holiday[features_with_holiday],
        np.log1p(valid_holiday["sales"])
    ),
    early_stopping_rounds=50
)

holiday_prediction = np.expm1(
    holiday_model.predict(valid_holiday[features_with_holiday])
).clip(min=0)

holiday_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_holiday["sales"],
        holiday_prediction
    )
)

print("Lag only:             ", lag_rmsle)
print("Lag + stores:         ", store_rmsle)
print("Lag + stores + holiday:", holiday_rmsle)

### Holiday feature result

RMSLE improves again to **0.41914**, the best score on this validation window. The calendar information helps here, so I keep it for the experiment. Later, the Kaggle result reminds me that one good validation window is not enough proof.


### Oil price

Oil data has missing weekdays and no weekend rows, so I carry the latest known price forward. The score became worse, so I did not keep this feature.


In [ ]:
oil = pd.read_csv(
    data_dir / "oil.csv",
    parse_dates=["date"]
)

print("Shape:", oil.shape)
print("Date range:", oil["date"].min(), "to", oil["date"].max())
print("\nMissing:")
display(oil.isna().sum())

display(oil.head(10))
display(oil.tail(10))

In [ ]:
oil_daily = pd.DataFrame({
    "date": pd.date_range(
        oil["date"].min(),
        oil["date"].max(),
        freq="D"
    )
})

oil_daily = oil_daily.merge(
    oil,
    on="date",
    how="left",
    validate="one_to_one"
)

oil_daily["dcoilwtico"] = (
    oil_daily["dcoilwtico"]
    .ffill()
    .bfill()  # Only the first date has no previous value.
)

fit_oil = fit_holiday.merge(
    oil_daily,
    on="date",
    how="left",
    validate="many_to_one"
)

valid_oil = valid_holiday.merge(
    oil_daily,
    on="date",
    how="left",
    validate="many_to_one"
)

features_with_oil = features_with_holiday + ["dcoilwtico"]

assert len(fit_oil) == len(fit_holiday)
assert len(valid_oil) == len(valid_holiday)
assert fit_oil["dcoilwtico"].notna().all()
assert valid_oil["dcoilwtico"].notna().all()

print("Train shape:", fit_oil.shape)
print("Validation shape:", valid_oil.shape)

display(
    valid_oil[
        ["date", "dcoilwtico"]
    ].drop_duplicates().head(16)
)

In [ ]:
oil_model = CatBoostRegressor(
    **holiday_model.get_params()
)

oil_model.fit(
    fit_oil[features_with_oil],
    np.log1p(fit_oil["sales"]),
    cat_features=cat_features_with_store,
    eval_set=(
        valid_oil[features_with_oil],
        np.log1p(valid_oil["sales"])
    ),
    early_stopping_rounds=50
)

oil_prediction = np.expm1(
    oil_model.predict(valid_oil[features_with_oil])
).clip(min=0)

oil_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_oil["sales"],
        oil_prediction
    )
)

print("Lag + stores:                  ", store_rmsle)
print("Lag + stores + holiday:        ", holiday_rmsle)
print("Lag + stores + holiday + oil:  ", oil_rmsle)

### Oil feature result

RMSLE becomes worse at **0.42561**. Oil price may matter to the business, but this raw feature does not help this model. I remove it instead of keeping every available column.


### Do store columns still help?

I remove the store columns once to check whether they add useful information beyond the holiday flags.


In [ ]:
lag_holiday_features = features_with_lag + holiday_features

lag_holiday_model = CatBoostRegressor(
    **holiday_model.get_params()
)

lag_holiday_model.fit(
    fit_holiday[lag_holiday_features],
    np.log1p(fit_holiday["sales"]),
    cat_features=cat_features,
    eval_set=(
        valid_holiday[lag_holiday_features],
        np.log1p(valid_holiday["sales"])
    ),
    early_stopping_rounds=50
)

lag_holiday_prediction = np.expm1(
    lag_holiday_model.predict(
        valid_holiday[lag_holiday_features]
    )
).clip(min=0)

lag_holiday_rmsle = np.sqrt(
    mean_squared_log_error(
        valid_holiday["sales"],
        lag_holiday_prediction
    )
)

print("Lag only:", lag_rmsle)
print("Lag + Holidays:", lag_holiday_rmsle)
print("Lag + Stores + Holidays:", holiday_rmsle)

## 7. Compare the experiments

Lower RMSLE is better. The extended model looked best on this single validation window.


In [ ]:
score_comparison = pd.DataFrame({
    "model": [
        "Baseline",
        "CatBoost",
        "Hybrid",
        "Lag 28",
        "Lag 28 + Stores",
        "Lag 28 + Holidays",
        "Lag 28 + Stores + Holidays",
        "Lag 28 + Stores + Holidays + Oil"
    ],
    "RMSLE": [
        baseline_rmsle,
        model_rmsle,
        hybrid_rmsle,
        lag_rmsle,
        store_rmsle,
        lag_holiday_rmsle,
        holiday_rmsle,
        oil_rmsle
    ]
}).sort_values("RMSLE")

display(score_comparison)

ax = score_comparison.plot.barh(
    x="model",
    y="RMSLE",
    figsize=(10, 5),
    legend=False,
    color="steelblue"
)

ax.set_title("Validation RMSLE Comparison (Lower is Better)")
ax.set_xlabel("RMSLE")
ax.set_ylabel("")

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3)

plt.show()

## 8. An important lesson

The extended model reached **0.41914 validation RMSLE**, but its Kaggle public score was **0.57279**, worse than the first hybrid model.

Possible reasons include choosing features on one 16-day validation window or a difference between my validation and final prediction pipeline. One public score cannot prove the exact cause. A better next step would be rolling validation across several 16-day windows and one shared pipeline for validation and test.

I keep the simpler hybrid as the final submission because it has the better Kaggle result. This experiment taught me that a better local score does not always mean a better future forecast.


## 9. Train the chosen model and create the submission

I retrain the hybrid approach using all available labeled data, check the IDs and predictions, then save `submission.csv`.


In [ ]:
final_train = train.copy()
model_test = test.copy()

for df in [final_train, model_test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

final_cutoff = final_train["date"].max() - pd.Timedelta(days=730)
final_fit = final_train[
    final_train["date"] >= final_cutoff
].copy()

final_model = CatBoostRegressor(
    iterations=model.get_best_iteration() + 1,
    depth=8,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

final_model.fit(
    final_fit[features],
    np.log1p(final_fit["sales"]),
    cat_features=cat_features
)

test_catboost_prediction = np.expm1(
    final_model.predict(model_test[features])
).clip(min=0)

In [ ]:
final_baseline_window = final_train[
    final_train["date"] >
    final_train["date"].max() - pd.Timedelta(days=28)
].copy()

final_baseline_table = (
    final_baseline_window
    .groupby(["store_nbr", "family", "day_of_week"])["sales"]
    .mean()
    .rename("baseline_prediction")
    .reset_index()
)

test_predictions = model_test[
    ["id", "store_nbr", "family", "day_of_week", "onpromotion"]
].copy()

test_predictions["catboost_prediction"] = test_catboost_prediction

test_predictions = test_predictions.merge(
    final_baseline_table,
    on=["store_nbr", "family", "day_of_week"],
    how="left",
    validate="many_to_one"
)

test_predictions["sales"] = np.where(
    test_predictions["onpromotion"] > 0,
    test_predictions["catboost_prediction"],
    test_predictions["baseline_prediction"]
)

sample_submission = pd.read_csv(
    data_dir / "sample_submission.csv"
)

submission = sample_submission[["id"]].merge(
    test_predictions[["id", "sales"]],
    on="id",
    how="left",
    validate="one_to_one"
)

assert len(submission) == len(test)
assert submission.columns.tolist() == ["id", "sales"]
assert submission["id"].equals(sample_submission["id"])
assert submission["sales"].notna().all()
assert (submission["sales"] >= 0).all()

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Submission shape:", submission.shape)
print("Missing:", submission["sales"].isna().sum())
print("Negative:", (submission["sales"] < 0).sum())
display(submission.head())

### Final submission result

The file has **28,512 predictions**, with no missing or negative values. This hybrid submission recorded **0.50406 public RMSLE** on Kaggle, which is better than my extended model submission.


## Final takeaway

My best public Kaggle score in this notebook is **0.50406 RMSLE**.

The biggest improvement came from using different prediction rules for promotional and non-promotional rows. The biggest lesson was validation: for time series, one future window is not enough when we keep testing many ideas on it.
